In [1]:
import pandas as pd
import numpy as np
import datetime



In [369]:
df_aemet=pd.read_csv(r"C:\Users\jaime\Desktop\Jaime_LLorca\proyectos\prediccion-electrica\data\interim\tabla_AEMT", encoding="utf-8")
df_omie=pd.read_csv(r"C:\Users\jaime\Desktop\Jaime_LLorca\proyectos\prediccion-electrica\data\interim\tabla_OMIE", encoding="utf-8")
df_esios_pred=pd.read_csv(r"C:\Users\jaime\Desktop\Jaime_LLorca\proyectos\prediccion-electrica\data\interim\tabla_ESIOS", encoding="utf-8")
df_esios_expl=pd.read_csv(r"C:\Users\jaime\Desktop\Jaime_LLorca\proyectos\prediccion-electrica\data\interim\tabla_ESIOS_explicativo", encoding="utf-8")

In [370]:
df_aemet.columns.tolist()
df_aemet.head()


,fecha,indicativo,nombre,provincia,altitud,tmed,prec,tmin,horatmin,tmax,...,horaPresMax,presMin,horaPresMin,hrMedia,hrMax,horaHrMax,hrMin,horaHrMin,pintMax,horaPIntMax
0,2023-01-01,0076,BARCELONA AEROPUERTO,BARCELONA,4,"12,2","0,0","8,6",07:22,"15,9",...,Varias,"1023,9",15,82.0,96.0,Varias,63.0,10:35,"0,0",NaN
1,2023-01-02,0076,BARCELONA AEROPUERTO,BARCELONA,4,"11,0","0,0","7,0",06:48,"15,0",...,Varias,"1022,8",05,61.0,86.0,00:00,47.0,13:54,"0,0",NaN
2,2023-01-03,0076,BARCELONA AEROPUERTO,BARCELONA,4,"12,9","0,0","8,2",23:30,"17,6",...,Varias,"1025,9",01,77.0,89.0,18:00,60.0,12:20,"0,0",NaN
3,2023-01-04,0076,BARCELONA AEROPUERTO,BARCELONA,4,"11,8","0,0","7,5",07:08,"16,1",...,Varias,"1031,2",Varias,76.0,89.0,Varias,57.0,12:47,"0,0",NaN
4,2023-01-05,0076,BARCELONA AEROPUERTO,BARCELONA,4,"10,6","0,0","5,7",07:35,"15,6",...,00,"1024,8",24,74.0,82.0,18:14,49.0,12:36,"0,0",NaN


In [371]:
# Claves + las 6 variables que nos quedamos
columnas_interes = [
    'fecha',        # clave temporal
    'nombre',   # clave de estación (para el pivot)
    'tmed',         # temperatura media  -> demanda
    'tmax',         # temperatura máxima  -> picos de demanda (verano)
    'tmin',         # temperatura mínima  -> picos de demanda (invierno)ç
    'velmedia',     # viento medio        -> generación eólica
    'racha',        # racha máxima        -> eventos de viento
    'sol',          # horas de sol        -> generación solar
    'prec',         # precipitación       -> hidráulica / nubosidad
]

df_aemet = df_aemet[columnas_interes]



In [372]:
df_aemet= df_aemet.pivot(index='fecha', columns='nombre', values=['tmed', 'tmax', 'velmedia', 'racha', 'sol', 'prec'])

In [373]:
import unicodedata

def limpia(txt):
    txt = txt.lower().replace(',', '').replace(' ', '_')
    txt = ''.join(c for c in unicodedata.normalize('NFKD', txt)
                  if not unicodedata.combining(c))
    return txt

df_aemet.columns = [f"{var}_{limpia(est)}" for var, est in df_aemet.columns]
df_aemet = df_aemet.reset_index()

In [374]:
df_aemet.columns.tolist() # ['fecha', 'tmed_alcazar_de_san_juan', ...]
df_aemet.shape              # (1277, 97)

(1277, 97)

In [375]:
cobertura = (df_aemet.isna().mean() * 100).round(2).sort_values(ascending=False)

# 1) las 100% vacías (las que se tiran seguro)
print("Columnas al 100% NaN:")
print(cobertura[cobertura == 100])

# 2) las que tienen huecos parciales (0 < NaN < 100) -> se imputan luego
print("\nHuecos parciales:")
print(cobertura[(cobertura > 0) & (cobertura < 100)])

Columnas al 100% NaN:
sol_puertollano            100.0
sol_valmadrid              100.0
sol_ecija                  100.0
sol_carmona                100.0
sol_miranda_de_ebro        100.0
sol_trujillo               100.0
sol_sarinena               100.0
sol_alcazar_de_san_juan    100.0
sol_almudevar              100.0
sol_san_pedro_manrique     100.0
dtype: float64

Huecos parciales:
prec_medina_de_pomar            19.97
prec_miranda_de_ebro            10.96
sol_medina_de_pomar              9.01
tmed_medina_de_pomar             8.61
tmax_medina_de_pomar             8.38
                                ...  
sol_madrid_aeropuerto            0.16
tmax_valencia_aeropuerto         0.08
velmedia_valencia_aeropuerto     0.08
velmedia_puertollano             0.08
tmed_valencia_aeropuerto         0.08
Length: 74, dtype: float64


In [376]:
muertas = df_aemet.columns[df_aemet.isna().mean() == 1.0].tolist()
df_aemet = df_aemet.drop(columns=muertas)

print(df_aemet.shape)          # 87 si no añadiste tmin; 103 si sí la añadiste
df_aemet.isna().mean().max()   # debe ser < 0.20 (ya no queda ningún 100%)

(1277, 87)


np.float64(0.19968676585747847)

In [377]:
df_aemet['fecha'] = pd.to_datetime(df_aemet['fecha'])
df_aemet['fecha'].dtype

dtype('<M8[ns]')

In [378]:
muertas = df_aemet.columns[df_aemet.isna().mean() == 1.0].tolist()
df_aemet = df_aemet.drop(columns=muertas)
df_aemet.shape   # esperado: (1277, 87)  -> 97 - 10

(1277, 87)

In [379]:
df_omie.loc[(df_omie.ano==2023)&(df_omie.mes==10)&(df_omie.dia==29),'hora'].max()  # ¿25?
df_omie.loc[(df_omie.ano==2023)&(df_omie.mes==3)&(df_omie.dia==26),'hora'].max()    # ¿23?                         # ¿empieza en 1 (1-based) o en 0?

np.int64(23)

In [380]:

# 1. día local a medianoche (desde ano/mes/dia)
dia = pd.to_datetime(df_omie[['ano','mes','dia']].rename(
        columns={'ano':'year','mes':'month','dia':'day'}))

# 2. ancla UTC (medianoche local -> UTC)
ancla = dia.dt.tz_localize('Europe/Madrid').dt.tz_convert('UTC')

# 3. paso por día: 60 min si el día tiene <=25 periodos, si no 15 min
n = df_omie.groupby(['ano','mes','dia'])['hora'].transform('size')
paso = np.where(n <= 25, 60, 15)

# 4. timestamp real = ancla + (hora-1)*paso, como tiempo transcurrido
df_omie['datetime_utc'] = ancla + pd.to_timedelta((df_omie['hora']-1)*paso, unit='m')


In [381]:
df_omie.groupby(['ano','mes','dia']).size().value_counts()   # verás 24, 96 y algún 23/25/92/100
df_omie['datetime_utc'].duplicated().sum()                   # 0
df_omie.loc[0, 'datetime_utc']                               # 2022-12-31 23:00:00+00:00

Timestamp('2022-12-31 23:00:00+0000', tz='UTC')

In [382]:
df_omie['datetime_utc'] = df_omie['datetime_utc'].dt.floor('h')

df_omie = (df_omie.groupby('datetime_utc')[['precio_espana','precio_portugal']]
                 .mean()
                 .reset_index())

In [383]:
df_omie['datetime_utc'].duplicated().sum()     # 0 -> una fila por hora
df_omie.shape                              # ~ nº de horas de la serie
df_omie.head()
df_omie.tail()

,datetime_utc,precio_espana,precio_portugal
30570,2026-06-30 17:00:00+00:00,65.8000,65.8000
30571,2026-06-30 18:00:00+00:00,112.7725,112.7725
30572,2026-06-30 19:00:00+00:00,133.8900,133.8900
30573,2026-06-30 20:00:00+00:00,131.3425,131.3425
30574,2026-06-30 21:00:00+00:00,123.7950,123.7950


In [384]:
print(df_omie['datetime_utc'].min(), df_omie['datetime_utc'].max())
print(df_esios_pred['datetime_utc'].min(), df_esios_pred['datetime_utc'].max())
print(df_esios_expl['datetime_utc'].min(), df_esios_expl['datetime_utc'].max())


2022-12-31 23:00:00+00:00 2026-06-30 21:00:00+00:00
2022-12-31 23:00:00+00:00 2026-06-29 22:00:00+00:00
2022-12-31 23:00:00+00:00 2026-06-29 22:00:00+00:00


In [385]:
df_esios_pred['datetime_utc'].duplicated().sum()   # 0
df_esios_expl['datetime_utc'].duplicated().sum()   # 0  (aplica el mismo fix a expl si no lo hiciste)

np.int64(0)

In [386]:
df_esios_pred['datetime_utc'] = pd.to_datetime(df_esios_pred['datetime_utc'], utc=True)
df_esios_expl['datetime_utc'] = pd.to_datetime(df_esios_expl['datetime_utc'], utc=True)


In [387]:
df_esios_pred['datetime_utc'].duplicated().sum()   # 
0
df_esios_expl['datetime_utc'].duplicated().sum()

np.int64(0)

In [388]:
df_esios_expl.shape


(30622, 9)

In [389]:
tabla_final = df_omie.merge(df_esios_pred, on='datetime_utc', how='left')
tabla_final = tabla_final.merge(df_esios_expl, on='datetime_utc', how='left')


In [390]:
# Convertir 'fecha' a medianoche en zona horaria Europe/Madrid y hacer naive
tabla_final['fecha'] = (tabla_final['datetime_utc']
                                .dt.tz_convert('Europe/Madrid')
                                .dt.normalize()
                                .dt.tz_localize(None))

In [391]:
print(len(df_omie), len(tabla_final))

30575 30575


In [394]:
tabla_final['fecha'] = (tabla_final['datetime_utc']
                        .dt.tz_convert('Europe/Madrid')   # UTC -> Madrid
                        .dt.normalize()                   # a medianoche del día local
                        .dt.tz_localize(None))            # naive, para casar con AEMET

In [395]:
tabla_final = tabla_final.merge(df_aemet, on='fecha', how='left')

In [396]:

print(len(tabla_final))                    # sigue 30575 (AEMET es única por fecha, no infla)
tabla_final['tmed_madrid_aeropuerto'].notna().sum()   # >0 -> AEMET se pegó
tabla_final[['datetime_utc','fecha','tmed_madrid_aeropuerto']].head(30)   # ver el broadcast: 24 h con el mismo valor

30575


,datetime_utc,fecha,tmed_madrid_aeropuerto
0,2022-12-31 23:00:00+00:00,2023-01-01,"10,2"
1,2023-01-01 00:00:00+00:00,2023-01-01,"10,2"
2,2023-01-01 01:00:00+00:00,2023-01-01,"10,2"
3,2023-01-01 02:00:00+00:00,2023-01-01,"10,2"
4,2023-01-01 03:00:00+00:00,2023-01-01,"10,2"
5,2023-01-01 04:00:00+00:00,2023-01-01,"10,2"
6,2023-01-01 05:00:00+00:00,2023-01-01,"10,2"
7,2023-01-01 06:00:00+00:00,2023-01-01,"10,2"
8,2023-01-01 07:00:00+00:00,2023-01-01,"10,2"
9,2023-01-01 08:00:00+00:00,2023-01-01,"10,2"


In [397]:
tabla_final.shape

(30575, 104)

In [402]:
tabla_final.to_parquet('../data/interim/tabla_maestra_estructural.parquet', index=False)